In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *


### **DATA READING**

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@customersprojectete.dfs.core.windows.net/orders")
df.count()

In [0]:
df.limit(5).display()

### # deleting the _rescued_column

In [0]:
df = df.drop("_rescued_data")

In [0]:
df  =df.dropDuplicates()

In [0]:
df = df.withColumn("year", year(col("order_date")))
df.limit(5).display()

In [0]:
window = Window.partitionBy("year").orderBy("total_amount")
df_total =df.groupBy(col("year"))\
            .agg(sum(col("total_amount")).alias("total_amount"))

df_total.limit(5).display()


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .save("abfss://silver@customersprojectete.dfs.core.windows.net/orders")

In [0]:
%sql
create table if not exists databricks_cata.silver.orders_silver
using delta
location "abfss://silver@customersprojectete.dfs.core.windows.net/orders"

In [0]:
%sql
select * from databricks_cata.silver.orders_silver